# L'hypothèse qui ne tient plus · *The assumption that no longer holds*

Notebook compagnon du chapitre **11. La croissance économique et son poids sur vos rendements de long terme** — [lire l'article](https://nmlab.io/ressources/croissance-economique-rendements-long-terme).
Companion notebook to chapter **11. Economic Growth and What It Is Worth to Your Long-Term Returns** — [read the article](https://nmlab.io/en/ressources/economic-growth-long-term-returns).

**Exécutez l'unique cellule ci-dessous** (bouton ▶ ou Ctrl+Entrée) : la figure se régénère avec les **données FRED du jour**. Passez `LANG = "en"` en tête de cellule pour les libellés anglais. — Run the single cell below (▶ or Ctrl+Enter) to rebuild the figure with **today's FRED data**; set `LANG = "en"` at the top for English labels.

Code : licence MIT · © 2026 [NMLab](https://nmlab.io) · dépôt [nmlab-finance/nmlab-figures](https://github.com/nmlab-finance/nmlab-figures)

In [ ]:
LANG = "fr"   # "fr" ou "en" — langue des libellés / label language

# Récupère puis active le style partagé NMLab (thème sombre + police Inter).
# Fetch and activate the shared NMLab style (dark theme + Inter font).
import urllib.request

urllib.request.urlretrieve("https://raw.githubusercontent.com/nmlab-finance/nmlab-figures/main/nmlab_style.py", "nmlab_style.py")
import nmlab_style as nm

nm.setup()


from pandas import Series

def load_profit_share() -> Series:
    """Part des profits des sociétés après impôt (IVA et CCAdj inclus) dans le PIB,
    en direct depuis FRED : CPATAX (profits après impôt) rapporté à GDP (PIB nominal), en %.
    U.S. after-tax corporate profit share of GDP, live from FRED (CPATAX over GDP)."""
    profits = nm.load_fred("CPATAX")
    gdp = nm.load_fred("GDP")
    return (profits / gdp * 100).dropna()

share = load_profit_share()


import matplotlib.dates as mdates
import pandas as pd
from matplotlib.figure import Figure

LABELS = {
    "fr": dict(
        title="L'hypothèse qui ne tient plus",
        sub="Part des profits des sociétés américaines après impôt dans le PIB",
        ylab="profits après impôt, % du PIB",
        pre="moyenne {a}-1999 : {v} %",
        post="moyenne depuis 2000 : {v} %",
        record="record : {v} %",
        note="Toute la mécanique de la dilution suppose cette part STABLE. Elle est passée d'environ 6 % à 9 % en\n"
             "moyenne depuis 2000, record fin 2025. Source : BEA via FRED (CPATAX — IVA et CCAdj inclus ; GDP)."),
    "en": dict(
        title="The assumption that no longer holds",
        sub="U.S. after-tax corporate profit share of GDP",
        ylab="after-tax profits, % of GDP",
        pre="average {a}-1999: {v}%",
        post="average since 2000: {v}%",
        record="record: {v}%",
        note="The whole dilution mechanism assumes this share is STABLE. It rose from about 6% to 9% on average\n"
             "since 2000, a record in late 2025. Source: BEA via FRED (CPATAX — incl. IVA and CCAdj; GDP)."),
}

def num(value: float, lang: str, dec: int = 1) -> str:
    """Nombre localisé (virgule décimale en français) / localized number."""
    s = f"{value:.{dec}f}"
    return s.replace(".", ",") if lang == "fr" else s

def build_figure(share: Series, lang: str) -> Figure:
    """Part des profits (trimestrielle) avec les moyennes avant/après 2000 et le record."""
    text = LABELS[lang]
    split = pd.Timestamp("2000-01-01")
    pre, post = share.loc[:"1999"], share.loc["2000":]
    fig = nm.figure(height_px=1045)
    ax = nm.axes(fig)
    ax.plot(share.index, share, color=nm.COLORS["blue"], linewidth=2.0)
    ax.plot([share.index[0], split], [pre.mean(), pre.mean()],
            color=nm.COLORS["text"], linestyle=(0, (7, 5)), linewidth=2.4)
    ax.plot([split, share.index[-1]], [post.mean(), post.mean()],
            color=nm.COLORS["rose"], linestyle=(0, (7, 5)), linewidth=2.4)
    ax.set_ylim(2, 12.6)
    ax.set_yticks(range(2, 13, 2))
    ax.set_ylabel(text["ylab"])
    ax.margins(x=0.01)
    ax.xaxis.set_major_locator(mdates.YearLocator(10))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.text(0.03, 0.94, text["pre"].format(a=share.index[0].year, v=num(pre.mean(), lang)),
            transform=ax.transAxes, fontsize=23, fontweight="bold", color=nm.COLORS["text"], va="top")
    ax.text(0.03, 0.865, text["post"].format(v=num(post.mean(), lang)),
            transform=ax.transAxes, fontsize=23, fontweight="bold", color=nm.COLORS["rose"], va="top")
    last_x, last_y = share.index[-1], share.iloc[-1]
    ax.annotate(text["record"].format(v=num(last_y, lang, 2)),
                xy=(last_x, last_y), xytext=(-260, 30), textcoords="offset points",
                ha="left", va="center", fontsize=25, fontweight="bold", color=nm.COLORS["amber"],
                arrowprops=dict(arrowstyle="->", color=nm.COLORS["amber"], lw=1.8,
                                connectionstyle="arc3,rad=-0.3"))
    nm.header(fig, text["title"], text["sub"])
    nm.footer(fig, text["note"])
    return fig

build_figure(share, LANG)